In [1]:
import os
from torch.utils.data import Dataset
from PIL import Image

class ImageNetValDataset(Dataset):
    def __init__(self, img_dir, img_to_label, transform=None):
        self.img_dir = img_dir
        self.img_to_label = img_to_label
        self.img_filenames = sorted(os.listdir(self.img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.img_filenames)

    def __getitem__(self, idx):
        img_filename = self.img_filenames[idx]
        img_path = os.path.join(self.img_dir, img_filename)
        print(img_path)
        image = Image.open(img_path).convert('RGB')
        label = self.img_to_label[idx] - 1 # indices should be from 0 to 999
        if self.transform:
            image = self.transform(image)
        return image, label

In [1]:
import torch
import torchvision
from torchvision import transforms
import torchvision.models as models
from torch.utils.data import DataLoader

transforms_for_models = {
    'alexnet':
        transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
    'inception_v3': 
        transforms.Compose([
            transforms.Resize(299),
            transforms.CenterCrop(299),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
    'vgg19': 
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
    'efficientnet-b0': 
        transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225])
])
}

datasets = {key: torchvision.datasets.ImageNet(root="./imagenet", split="val", transform=transforms_for_models[key])
                    for key in transforms_for_models}
dataloaders = {key: DataLoader(datasets[key], batch_size=32, shuffle=False) for key in datasets}

models_to_test = {
    'alexnet': models.alexnet(pretrained=True),
    'inception_v3': models.inception_v3(pretrained=True),
    'vgg19': models.vgg19(pretrained=True),
    'efficientnet-b0': models.efficientnet_b0(pretrained=True)
}


/opt/anaconda3/envs/classification_performance/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/classification_performance/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/opt/anaconda3/envs/classification_performance/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_

In [2]:
import timm
import torch

# For ViT-Small (patch size 16)
model_dino = timm.create_model('vit_small_patch16_224_dino', pretrained=True)
model_dino.eval()

/opt/anaconda3/envs/classification_performance/lib/python3.12/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name vit_small_patch16_224_dino to current vit_small_patch16_224.dino.
  model = create_fn(


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity(

In [3]:
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform_dino = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

dataset_dino = torchvision.datasets.ImageNet(root="./imagenet", split="val", transform=transform_dino)
dataloader_dino = DataLoader(dataset_dino, batch_size=64, shuffle=False, num_workers=4)

In [4]:
all_features = []
all_labels = []
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model_dino = model_dino.to(device)

with torch.no_grad():
    for images, labels in dataloader_dino:
        images = images.to(device)
        labels = labels.to(device)
        features = model_dino.forward_features(images)  # shape: [batch, 384]
        all_features.append(features.cpu())
        all_labels.append(labels)

features = torch.cat(all_features)
labels = torch.cat(all_labels)
print(features.shape)
print(labels.shape)
torch.save({'features': features, 'labels': labels}, 'models/features_labels.pt')

torch.Size([50000, 197, 384])
torch.Size([50000])


In [1]:
import torch
checkpoint = torch.load('models/features_labels.pt')
features = checkpoint['features']
labels = checkpoint['labels']

print(features.shape)
print(labels.shape)

torch.Size([50000, 197, 384])
torch.Size([50000])


In [7]:
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga')
clf.fit(X, y)

preds = clf.predict(X)
acc = accuracy_score(y, preds)
print(f"Validation Accuracy: {acc:.2%}")

ValueError: Found array with dim 3. LogisticRegression expected <= 2.

In [64]:
import torch
import torchvision.models as models

# Move each model to the appropriate device
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
for name, model in models_to_test.items():
    models_to_test[name] = model.to(device)

In [65]:
import torch.nn as nn
import torch.nn.functional as F

# Set each model to evaluation mode
for model in models_to_test.values():
    model.eval()

# Initialize metrics
total_loss = {key: 0.0 for key in models_to_test}
correct = {key: 0 for key in models_to_test}
total = {key: 0 for key in models_to_test}

# Define the loss function
criterion = nn.CrossEntropyLoss(reduction='sum')

# Disable gradient computation for evaluation
with torch.no_grad():
    for name, model in models_to_test.items():
        processed = 0
        for images, labels in dataloaders[name]:
            images = images.to(device)
            labels = labels.to(device)
    
            # Forward pass
            outputs = model(images)
            _, index = outputs.max(1)
    
            # Compute loss
           
            loss = criterion(outputs, labels)        
            total_loss[name] += loss
    
            # Compute accuracy
            _, predicted = outputs.max(1)
            correct[name] += predicted.eq(labels).sum().item()
            total[name] += labels.size(0)

            processed += images.size(0)
            if processed % 10000 < images.size(0):
                print(f'Model: {name}, Processed {processed} / 50000 images')

# Calculate average loss and accuracy
for name, model in models_to_test.items():
    avg_loss = total_loss[name] / total[name]
    accuracy = 100 * correct[name] / total[name]
    print(f'Model: {name}, Training Set: Imagenet')
    print(f'Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')
    print('__________________________________________________________')

Model: alexnet, Processed 10016 / 50000 images
Model: alexnet, Processed 20000 / 50000 images
Model: alexnet, Processed 30016 / 50000 images
Model: alexnet, Processed 40000 / 50000 images
Model: alexnet, Processed 50000 / 50000 images
Model: inception_v3, Processed 10016 / 50000 images
Model: inception_v3, Processed 20000 / 50000 images
Model: inception_v3, Processed 30016 / 50000 images
Model: inception_v3, Processed 40000 / 50000 images
Model: inception_v3, Processed 50000 / 50000 images
Model: vgg19, Processed 10016 / 50000 images
Model: vgg19, Processed 20000 / 50000 images
Model: vgg19, Processed 30016 / 50000 images
Model: vgg19, Processed 40000 / 50000 images
Model: vgg19, Processed 50000 / 50000 images
Model: efficientnet-b0, Processed 10016 / 50000 images
Model: efficientnet-b0, Processed 20000 / 50000 images
Model: efficientnet-b0, Processed 30016 / 50000 images
Model: efficientnet-b0, Processed 40000 / 50000 images
Model: efficientnet-b0, Processed 50000 / 50000 images
Model

In [ ]:
"""
Model: alexnet, Training Set: Imagenet
Validation Loss: 1.9096, Accuracy: 56.5560
__________________________________________________________
Model: inception_v3, Training Set: Imagenet
Validation Loss: 0.9895, Accuracy: 77.2140
__________________________________________________________
Model: vgg19, Training Set: Imagenet
Validation Loss: 1.1920, Accuracy: 70.6700
__________________________________________________________
Model: efficientnet-b0, Training Set: Imagenet
Validation Loss: 0.9571, Accuracy: 77.6720 

"""